In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## IgA categorization

IgA_Absolute is a column which contains the values of the IgA obtained through a blood analysis.
The original column contains "object" values, i.e. strings --> some values are numerical and others are indicated as > or < of a given value. Therefore, they must be modified.

The categorization, according to the physicians' indication, is to distinguish whether:
- Class NEG = Negative values (very low (<20 (mg/dL))/not detected/Selective_deficit_IgA = 1)
- Class NORMAL = Normal values (between 20 and 10X threshold)
- Class HIGH = High values (>10X threshold)

where threshold = 10 mg/dL

In [2]:
# IMPORT OF THE LAST VERSION OF THE DATASET

df = pd.read_excel("Data_explorative_analysis_part1.xlsx")
Dataset = df.copy()

Before assignig the proper classes mentioned above, it is necessary to assign temporary classes to some values as we can see in the following lines of code

In [ ]:
Dataset['IgA_Absolute'][Dataset['IgA_Absolute'] == '<10X'] = -3
Dataset['IgA_Absolute'][Dataset['IgA_Absolute'] == '8, 2'] = 0
Dataset['IgA_Absolute'][Dataset['IgA_Absolute'] == '16,3'] = 0
Dataset['IgA_Absolute'][Dataset['IgA_Absolute'].str[0] == '>'] = 100
Dataset['IgA_Absolute'][Dataset['IgA_Absolute'] == 'X10N'] = 100

Dataset['IgA_Absolute'][Dataset['IgA_Absolute'] == "     "] = -2

Now all the values are numbers, but they still need to be converted in numerical type

In [4]:
Dataset['IgA_Absolute'] = pd.to_numeric(Dataset['IgA_Absolute'])

Final classes assignment:

In [5]:
IgA_threshold = 10      #mg/dl (threshold for which if IgA_abs > 10xthreshold --> High values = class HIGH)
IgA_lower_thresold = 20 #mg/dl (threshold for which if IgA_abs < lower_threshold --> Negative values = class NEG) --> see value in the diagnostic algorithm

conditions = [
    (Dataset['IgA_Absolute'] >= 10*IgA_threshold),
    (Dataset['IgA_Absolute'] == -3),
    ((((Dataset['IgA_Absolute'] == 0) | (Dataset['IgA_Absolute'] < IgA_lower_thresold)) & (Dataset['IgA_Absolute'] > 0)) | (Dataset['Selective_deficit_IgA'] == 1)),
    (Dataset['IgA_Absolute'] == -1),
    ((Dataset['IgA_Absolute'] == -2) | (Dataset['IgA_Absolute'].isna() == True))
]

choices =[
    'HIGH',
    'NORMAL',
    'NEG',
    '-1',
    pd.NA
]

Dataset['IgA_categorical'] = np.select(conditions, choices, default = 'NORMAL')

Dataset.to_excel("Data_explorative_control_IgA.xlsx", index=False)
np.sum(Dataset['IgA_categorical'].isna())

1042

The values corresponding to -1 (strange values) are manually modified in the excel file Data_explorative_control_IgA.xlsx based on their original values in the column IgA_Absolute in the original dataset (file excel Data_w_cat_children_NEW). 

The modified dataset is imported below

In [6]:
df = pd.read_excel("Data_explorative_control_IgA.xlsx")
Dataset = df.copy()

## EMA Categorization

- values = to 1 --> new class: POS
- values = to 0 --> new class: NEG
- if nan and IgA NORMAL or NEG --> new class: NP (Not Performed) - see diagnostic algorithm (directly goes to biopsy without perform EMA)
- if nan and IgA HIGH --> new class: NA (Not Available)

Before assignig the proper classes mentioned above, it is necessary to assign manually classes to some values as we can see in the following lines of code

In [ ]:
Dataset['EMA'][Dataset['EMA'] == 'IgG pos'] = 1
Dataset['EMA'][Dataset['EMA'] == '>100'] = 1
Dataset['EMA'][Dataset['EMA'] == '88'] = 1
Dataset['EMA'][Dataset['EMA'] == " "] = pd.NA
Dataset['EMA'][Dataset['EMA'] == "     "] = pd.NA
Dataset['EMA'][Dataset['EMA'] == "      "] = pd.NA

# conversion to numerical array
Dataset['EMA'] = pd.to_numeric(Dataset['EMA'])

np.sum(Dataset['EMA'].isna())

Assign the classes reported above:

In [8]:
conditions = [
    (Dataset['EMA'] == 1),
    (Dataset['EMA'] == 0),
    ((Dataset['EMA'].isna() == True) & ((Dataset['IgA_categorical'] == "NORMAL") | (Dataset['IgA_categorical'] == "NEG"))),
]

choices =[
    'POS',
    'NEG',
    'NP'
]

Dataset['EMA_categorical'] = np.select(conditions, choices, default = pd.NA)

np.sum(Dataset['EMA_categorical'].isna())

1003

## HLA Haplotype binarization

- HLA Haplotype DQ2 or DQ8 are present --> new class: 1
- HLA Haplotype missing or NA --> new class: 0

In [ ]:
Dataset['HLA_binary'] = 1
Dataset['HLA_binary'][(Dataset['HLA Haplotype'].isna() == True) | (Dataset['HLA Haplotype'] == "NA")] = 0

Dataset = Dataset.drop(columns='HLA Haplotype')

## MARSH

Two new classes were added to replace the missing values:

- Class NP = Not Performed, is assigned as indicated by the physician for the presence of other indicators that allows directly to confirm the diagnosis (the conditions are that used below in the code)

- Class NA = Not Available, assigned in all the other cases

In [ ]:
Dataset['MARSH_Histology'][((Dataset['Endoscopy_Duodenum'] == 'Not Performed') | 
                           ((Dataset['IgA_categorical'] == 'HIGH') & (Dataset['EMA_categorical'] == "POS")) |
                           ((Dataset['IgA_categorical'] == 'HIGH') & (Dataset['HLA_binary'] == 1)))
                           & ((Dataset['MARSH_Histology'].isna() == True))] = "NP"

Dataset['MARSH_Histology'] = Dataset['MARSH_Histology'].fillna("NA")

Dataset['MARSH_Histology'][Dataset['MARSH_Histology'] == "4"] = '3C' # since they were misclassified as 4 but Corazza_histology was B2


## Corazza

In [ ]:
Dataset['Corazza_Histology'][((Dataset['Endoscopy_Duodenum'] == 'Not Performed') | 
                           ((Dataset['IgA_categorical'] == 'HIGH') & (Dataset['EMA_categorical'] == "POS")) |
                           ((Dataset['IgA_categorical'] == 'HIGH') & (Dataset['HLA_binary'] == 1)))
                           & ((Dataset['Corazza_Histology'].isna() == True) | (Dataset['Corazza_Histology'] == 0))] = "NP"

Dataset['Corazza_Histology'] = Dataset['Corazza_Histology'].fillna("NA")


In [12]:
sum(Dataset['Corazza_Histology'].isna())

0

## Recover the missing values in IgA and EMA with a new class (NC = Not Classified), where Corazza is present

In [ ]:
Dataset['IgA_categorical'][(Dataset['IgA_categorical'].isna() == True) & (Dataset['Corazza_Histology'] != "NA")] = "NC"
Dataset['EMA_categorical'][(Dataset['EMA_categorical'].isna() == True) & (Dataset['Corazza_Histology'] != "NA")] = "NC"

## Further Pre-processing

Patients with missing EMA, HLA and Biopsy (Corazza Histology) has to be attentioned:

In [14]:
Dataset[(Dataset['EMA_categorical'].isna() == True) &  (Dataset['HLA_binary'] == 0) & (Dataset['Corazza_Histology'] == "NA")]

,Patient_ID,Local_ID,Center,Sex,Ethnicity,Year_of_Diagnosis,Age_at_Diagnosis,Age_Years,Age_Months,Month_Gluten_Started,...,IgG_10X,IgG_10X_Categoriale,Endoscopy_Duodenum,MARSH_Histology,Corazza_Histology,Biagi_Diet_Adherence,VAS_Score_Adherence,IgA_categorical,EMA_categorical,HLA_binary
357,367,BA367,BA,1.0,1.0,2018,9.300000,9,3.000000,6.0,...,NaN,NaN,NaN,NA,NA,NaN,10,HIGH,<NA>,0
1060,1105,NA32,NaN,1.0,1.0,2010,1.000000,1,0.000000,NaN,...,NaN,NaN,NaN,NA,NA,4,NaN,NaN,<NA>,0
1063,1108,NA35,NaN,1.0,1.0,2014,6.000000,6,0.000000,NaN,...,NaN,NaN,NaN,NA,NA,4,NaN,NaN,<NA>,0
1073,1118,NA45,NaN,1.0,1.0,2016,14.000000,14,0.000000,NaN,...,NaN,NaN,NaN,NA,NA,4,NaN,NaN,<NA>,0
1076,1121,NA48,NaN,0.0,1.0,2018,4.000000,4,0.000000,5.0,...,NaN,NaN,NaN,NA,NA,4,NaN,NaN,<NA>,0
1135,1181,NA108,NaN,1.0,1.0,2017,5.000000,5,0.000000,NaN,...,NaN,NaN,NaN,NA,NA,4,NaN,NaN,<NA>,0
1615,1679,NAV204,NaN,1.0,1.0,2015,14.000000,14,0.000000,NaN,...,<10x,0.0,NaN,NA,NA,NaN,NaN,NaN,<NA>,0
2895,2999,RMS166,RM,1.0,1.0,2014,17.700000,17,7.000000,6.0,...,NaN,NaN,NaN,NA,NA,NaN,NaN,NaN,<NA>,0
2935,3039,RMS206,RM,0.0,1.0,2014,9.100000,9,1.000000,6.0,...,NaN,NaN,NaN,NA,NA,NaN,NaN,HIGH,<NA>,0
2945,3049,RMS216,RM,0.0,1.0,2014,6.500000,6,5.000000,6.0,...,NaN,NaN,NaN,NA,NA,NaN,NaN,NaN,<NA>,0


Since these 45 rows respect the requisits above specified and have many missing values, we drop them

In [ ]:
i = Dataset[(Dataset['EMA_categorical'].isna() == True) &  (Dataset['HLA_binary'] == 0) & (Dataset['Corazza_Histology'] == "NA")].index
Dataset = Dataset.drop(i)

Drop useless columns:

In [16]:
Dataset.columns

Index(['Patient_ID', 'Local_ID', 'Center', 'Sex', 'Ethnicity',
       'Year_of_Diagnosis', 'Age_at_Diagnosis', 'Age_Years', 'Age_Months',
       'Month_Gluten_Started', 'Weight', 'Height', 'Growth Delay Updated',
       'BMI', 'Recent_Gastroenteritis', 'Num_Siblings', 'Blood_Count',
       'BloodCount_Microcytic_Anemia', 'GI_Symptoms', 'GI_Upper',
       'GI_Diarrhea', 'GI_WeightLoss', 'GI_Meteorism', 'GI_AbdominalPain',
       'GI_Anorexia', 'GI_Stipsis', 'GI_Other', 'Neurological_Sympt',
       'Neuro_Psychiatry', 'Neuro_Other', 'Neuro_Headache',
       'Neuro_Irritability', 'Delayed_Menarche', 'Teeth_Anomalies',
       'Blood_Proteins_Anomalies', 'Asthenia', 'Osteopenia', 'Autoimmune_Dis',
       'Autoimmune_Tyroid_Dis', 'Autoimmune_Type1_Diabetes',
       'Autoimmune_Skin/Tissues', 'Autoimmune_Others', 'Common_deficit_Ig',
       'Selective_deficit_IgA', 'Down_Syndrome', 'Dermatisis_Herpetiformis',
       'Familiarity', 'Months_from_symptoms', 'Reason_Diagnosis',
       'First_Diag

In [19]:
col = ['Local_ID', 'Center', 'Ethnicity',
       'Year_of_Diagnosis', 'Age_at_Diagnosis', 'Age_Months',
       'Month_Gluten_Started', 'Weight', 'Height',
       'BMI', 'Num_Siblings', 'Blood_Count',
        'GI_Symptoms', 'Neurological_Sympt',
       'Delayed_Menarche', 'Autoimmune_Dis', 'Months_from_symptoms', 'Reason_Diagnosis',
       'First_Diagnosis_Doctor', 'Diagnosis_modality',
       'Doctor-dependent_Delay', 'Total_Diagnostic_Delay', 'Wrong_Diagnosis',
       'Wrong_Diagnosis_2', 'Therapy_before_Diagn',
       'Complications', 'Complications_Hospitalization',
       'Complications_DayHospital', 'Anti-tTG', 'IgA_Absolute',
       'EMA', 'IgG_Absolute', 'IgG_10X', 'IgG_10X_Categoriale',
       'Endoscopy_Duodenum',
       'Biagi_Diet_Adherence', 'VAS_Score_Adherence']

Dataset = Dataset.drop(columns = col)


In [20]:
Dataset.dtypes

Patient_ID                        int64
Sex                             float64
Age_Years                         int64
Growth Delay Updated            float64
Recent_Gastroenteritis          float64
BloodCount_Microcytic_Anemia      int64
GI_Upper                          int64
GI_Diarrhea                       int64
GI_WeightLoss                     int64
GI_Meteorism                      int64
GI_AbdominalPain                  int64
GI_Anorexia                       int64
GI_Stipsis                        int64
GI_Other                          int64
Neuro_Psychiatry                  int64
Neuro_Other                       int64
Neuro_Headache                    int64
Neuro_Irritability                int64
Teeth_Anomalies                 float64
Blood_Proteins_Anomalies        float64
Asthenia                        float64
Osteopenia                      float64
Autoimmune_Tyroid_Dis             int64
Autoimmune_Type1_Diabetes         int64
Autoimmune_Skin/Tissues           int64


In [21]:
Dataset['Sex'] = Dataset['Sex'].astype('Int64')
Dataset['Growth Delay Updated'] = Dataset['Growth Delay Updated'].astype('Int64')
Dataset['Recent_Gastroenteritis'] = Dataset['Recent_Gastroenteritis'].astype('Int64')
Dataset['Teeth_Anomalies'] = Dataset['Teeth_Anomalies'].astype('Int64')
Dataset['Blood_Proteins_Anomalies'] = Dataset['Blood_Proteins_Anomalies'].astype('Int64')
Dataset['Asthenia'] = Dataset['Asthenia'].astype('Int64')
Dataset['Osteopenia'] = Dataset['Osteopenia'].astype('Int64')
Dataset['Common_deficit_Ig'] = Dataset['Common_deficit_Ig'].astype('Int64')
Dataset['Selective_deficit_IgA'] = Dataset['Selective_deficit_IgA'].astype('Int64')
Dataset['Down_Syndrome'] = Dataset['Down_Syndrome'].astype('Int64')
Dataset['Dermatisis_Herpetiformis'] = Dataset['Dermatisis_Herpetiformis'].astype('Int64')
Dataset['Familiarity'] = Dataset['Familiarity'].astype('Int64')

Dataset.dtypes

Patient_ID                       int64
Sex                              Int64
Age_Years                        int64
Growth Delay Updated             Int64
Recent_Gastroenteritis           Int64
BloodCount_Microcytic_Anemia     int64
GI_Upper                         int64
GI_Diarrhea                      int64
GI_WeightLoss                    int64
GI_Meteorism                     int64
GI_AbdominalPain                 int64
GI_Anorexia                      int64
GI_Stipsis                       int64
GI_Other                         int64
Neuro_Psychiatry                 int64
Neuro_Other                      int64
Neuro_Headache                   int64
Neuro_Irritability               int64
Teeth_Anomalies                  Int64
Blood_Proteins_Anomalies         Int64
Asthenia                         Int64
Osteopenia                       Int64
Autoimmune_Tyroid_Dis            int64
Autoimmune_Type1_Diabetes        int64
Autoimmune_Skin/Tissues          int64
Autoimmune_Others        

Analysis of the missing values row-wise:

In [22]:
Dataset.isna().sum()

Patient_ID                        0
Sex                               1
Age_Years                         0
Growth Delay Updated             13
Recent_Gastroenteritis            1
BloodCount_Microcytic_Anemia      0
GI_Upper                          0
GI_Diarrhea                       0
GI_WeightLoss                     0
GI_Meteorism                      0
GI_AbdominalPain                  0
GI_Anorexia                       0
GI_Stipsis                        0
GI_Other                          0
Neuro_Psychiatry                  0
Neuro_Other                       0
Neuro_Headache                    0
Neuro_Irritability                0
Teeth_Anomalies                 159
Blood_Proteins_Anomalies        153
Asthenia                          5
Osteopenia                      266
Autoimmune_Tyroid_Dis             0
Autoimmune_Type1_Diabetes         0
Autoimmune_Skin/Tissues           0
Autoimmune_Others                 0
Common_deficit_Ig               127
Selective_deficit_IgA       

We drop the rows where CLASSIFICATION is missing because they are observations for which the class labeling needed as groundtruth is not available

In [23]:
Dataset = Dataset.dropna(subset = 'CLASSIFICATION')

Print the new distribution of missing values:

In [24]:
Dataset.isna().sum()

Patient_ID                       0
Sex                              1
Age_Years                        0
Growth Delay Updated             0
Recent_Gastroenteritis           1
BloodCount_Microcytic_Anemia     0
GI_Upper                         0
GI_Diarrhea                      0
GI_WeightLoss                    0
GI_Meteorism                     0
GI_AbdominalPain                 0
GI_Anorexia                      0
GI_Stipsis                       0
GI_Other                         0
Neuro_Psychiatry                 0
Neuro_Other                      0
Neuro_Headache                   0
Neuro_Irritability               0
Teeth_Anomalies                 30
Blood_Proteins_Anomalies        23
Asthenia                         3
Osteopenia                      52
Autoimmune_Tyroid_Dis            0
Autoimmune_Type1_Diabetes        0
Autoimmune_Skin/Tissues          0
Autoimmune_Others                0
Common_deficit_Ig                1
Selective_deficit_IgA            1
Down_Syndrome       

Drop the row where Sex is not available:

In [26]:
Dataset = Dataset.dropna(subset = 'Sex')

We have replaced all the missing values with zero, except for IgA and EMA:

In [27]:
Dataset[['Recent_Gastroenteritis', 'Teeth_Anomalies', 'Blood_Proteins_Anomalies', 'Asthenia', 'Osteopenia', 'Common_deficit_Ig', 'Selective_deficit_IgA', 'Down_Syndrome', 'Dermatisis_Herpetiformis', 'Familiarity']] = Dataset[['Recent_Gastroenteritis', 'Teeth_Anomalies', 'Blood_Proteins_Anomalies', 'Asthenia', 'Osteopenia', 'Common_deficit_Ig', 'Selective_deficit_IgA', 'Down_Syndrome', 'Dermatisis_Herpetiformis', 'Familiarity']].fillna(value = 0)
Dataset.isna().sum()

Patient_ID                       0
Sex                              0
Age_Years                        0
Growth Delay Updated             0
Recent_Gastroenteritis           0
BloodCount_Microcytic_Anemia     0
GI_Upper                         0
GI_Diarrhea                      0
GI_WeightLoss                    0
GI_Meteorism                     0
GI_AbdominalPain                 0
GI_Anorexia                      0
GI_Stipsis                       0
GI_Other                         0
Neuro_Psychiatry                 0
Neuro_Other                      0
Neuro_Headache                   0
Neuro_Irritability               0
Teeth_Anomalies                  0
Blood_Proteins_Anomalies         0
Asthenia                         0
Osteopenia                       0
Autoimmune_Tyroid_Dis            0
Autoimmune_Type1_Diabetes        0
Autoimmune_Skin/Tissues          0
Autoimmune_Others                0
Common_deficit_Ig                0
Selective_deficit_IgA            0
Down_Syndrome       

Drop of the rows where IgA and EMA still remain missing:

In [28]:
Dataset = Dataset.dropna(subset = 'IgA_categorical')
Dataset = Dataset.dropna(subset = 'EMA_categorical')
Dataset.isna().sum().sum()

0

**Now, the dataset is complete!!!**

Dataset final dimensions:

In [29]:
Dataset.shape

(2922, 39)

## Save the Dataset

In [31]:
Dataset.columns

Index(['Patient_ID', 'Sex', 'Age_Years', 'Growth Delay Updated',
       'Recent_Gastroenteritis', 'BloodCount_Microcytic_Anemia', 'GI_Upper',
       'GI_Diarrhea', 'GI_WeightLoss', 'GI_Meteorism', 'GI_AbdominalPain',
       'GI_Anorexia', 'GI_Stipsis', 'GI_Other', 'Neuro_Psychiatry',
       'Neuro_Other', 'Neuro_Headache', 'Neuro_Irritability',
       'Teeth_Anomalies', 'Blood_Proteins_Anomalies', 'Asthenia', 'Osteopenia',
       'Autoimmune_Tyroid_Dis', 'Autoimmune_Type1_Diabetes',
       'Autoimmune_Skin/Tissues', 'Autoimmune_Others', 'Common_deficit_Ig',
       'Selective_deficit_IgA', 'Down_Syndrome', 'Dermatisis_Herpetiformis',
       'Familiarity', 'CLASSIFICATION', 'Complications_Iron_infusion',
       'Complications_Albumin', 'MARSH_Histology', 'Corazza_Histology',
       'IgA_categorical', 'EMA_categorical', 'HLA_binary'],
      dtype='object')

In [32]:
# REORDERING COLUMNS

Dataset = Dataset[['Patient_ID', 'Sex', 'Age_Years', 'Growth Delay Updated',
       'Recent_Gastroenteritis', 'BloodCount_Microcytic_Anemia', 'GI_Upper',
       'GI_Diarrhea', 'GI_WeightLoss', 'GI_Meteorism', 'GI_AbdominalPain',
       'GI_Anorexia', 'GI_Stipsis', 'GI_Other', 'Neuro_Psychiatry',
       'Neuro_Other', 'Neuro_Headache', 'Neuro_Irritability',
       'Teeth_Anomalies', 'Blood_Proteins_Anomalies', 'Asthenia', 'Osteopenia',
       'Autoimmune_Tyroid_Dis', 'Autoimmune_Type1_Diabetes',
       'Autoimmune_Skin/Tissues', 'Autoimmune_Others', 'Common_deficit_Ig',
       'Selective_deficit_IgA', 'Down_Syndrome', 'Dermatisis_Herpetiformis',
       'Familiarity', 'Complications_Iron_infusion',
       'Complications_Albumin', 'MARSH_Histology', 'Corazza_Histology',
       'IgA_categorical', 'EMA_categorical', 'HLA_binary', 'CLASSIFICATION']]



In [34]:
# SAVE DATASET

Dataset.to_excel("Data_explorative_analysis_part2.xlsx", index=False)

# Dataset preparation for TDA

In [51]:
df = pd.read_excel("Data_explorative_analysis_part2.xlsx")
Dataset = df.copy()

Drop of columns not used for TDA analysis

In [52]:
Dataset = Dataset.drop(columns = ['Complications_Iron_infusion', 'Complications_Albumin', 'MARSH_Histology'])
Dataset.columns

Index(['Patient_ID', 'Sex', 'Age_Years', 'Growth Delay Updated',
       'Recent_Gastroenteritis', 'BloodCount_Microcytic_Anemia', 'GI_Upper',
       'GI_Diarrhea', 'GI_WeightLoss', 'GI_Meteorism', 'GI_AbdominalPain',
       'GI_Anorexia', 'GI_Stipsis', 'GI_Other', 'Neuro_Psychiatry',
       'Neuro_Other', 'Neuro_Headache', 'Neuro_Irritability',
       'Teeth_Anomalies', 'Blood_Proteins_Anomalies', 'Asthenia', 'Osteopenia',
       'Autoimmune_Tyroid_Dis', 'Autoimmune_Type1_Diabetes',
       'Autoimmune_Skin/Tissues', 'Autoimmune_Others', 'Common_deficit_Ig',
       'Selective_deficit_IgA', 'Down_Syndrome', 'Dermatisis_Herpetiformis',
       'Familiarity', 'Corazza_Histology', 'IgA_categorical',
       'EMA_categorical', 'HLA_binary', 'CLASSIFICATION'],
      dtype='object')

Conversion of the variable Age_Years from dtype int64 to dtype float64

In [53]:
Dataset['Age_Years'] = Dataset['Age_Years'].astype('float64')

Conversion of all the categories in categorical variables ('Corazza_Histology', 'IgA_categorical', 'EMA_categorical', 'CLASSIFICATION') into numerical codes

- Corazza_Histology:

In [54]:
Dataset['Corazza_Histology'].unique()
Dataset.loc[Dataset['Corazza_Histology'].isna() == True, 'Corazza_Histology'] = 'N'
Dataset['Corazza_Histology'].unique()


array(['B2', 'NP', 'B1', 'N', 'A', 'b1'], dtype=object)

In [55]:
dictionary = {'A' : 0, 'B1' : 1, 'b1' : 1, 'B2' : 2, 'NP': 3, 'N' : 4}
Dataset['Corazza_Histology_NUM'] = Dataset['Corazza_Histology'].map(dictionary)
Dataset['Corazza_Histology_NUM'].unique()

array([2, 3, 1, 4, 0], dtype=int64)

- IgA_categorical:

In [56]:
Dataset['IgA_categorical'].unique()

array(['HIGH', 'NEG', 'NORMAL', 'NC'], dtype=object)

In [57]:
dictionary = {'NEG' : 0, 'NORMAL' : 1, 'HIGH' : 2, 'NC': 3}
Dataset['IgA_categorical_NUM'] = Dataset['IgA_categorical'].map(dictionary)
Dataset['IgA_categorical_NUM'].unique()

array([2, 0, 1, 3], dtype=int64)

- EMA_categorical:

In [58]:
Dataset['EMA_categorical'].unique()

array(['POS', 'NEG', 'NC', 'NP'], dtype=object)

In [59]:
dictionary = {'NEG' : 0, 'POS' : 1, 'NP' : 2, 'NC': 3}
Dataset['EMA_categorical_NUM'] = Dataset['EMA_categorical'].map(dictionary)
Dataset['EMA_categorical_NUM'].unique()

array([1, 0, 3, 2], dtype=int64)

- CLASSIFICATION:

In [60]:
Dataset['CLASSIFICATION'].unique()

array(['Major', 'Minor', 'Silent'], dtype=object)

In [61]:
dictionary = {'Silent' : 0, 'Minor' : 1, 'Major' : 2}
Dataset['CLASSIFICATION_NUM'] = Dataset['CLASSIFICATION'].map(dictionary)
Dataset['CLASSIFICATION_NUM'].unique()

array([2, 1, 0], dtype=int64)

Drop the duplicated columns:

In [62]:
Dataset = Dataset.drop(columns = ['Corazza_Histology', 'IgA_categorical', 'EMA_categorical', 'CLASSIFICATION'])
Dataset.columns

Index(['Patient_ID', 'Sex', 'Age_Years', 'Growth Delay Updated',
       'Recent_Gastroenteritis', 'BloodCount_Microcytic_Anemia', 'GI_Upper',
       'GI_Diarrhea', 'GI_WeightLoss', 'GI_Meteorism', 'GI_AbdominalPain',
       'GI_Anorexia', 'GI_Stipsis', 'GI_Other', 'Neuro_Psychiatry',
       'Neuro_Other', 'Neuro_Headache', 'Neuro_Irritability',
       'Teeth_Anomalies', 'Blood_Proteins_Anomalies', 'Asthenia', 'Osteopenia',
       'Autoimmune_Tyroid_Dis', 'Autoimmune_Type1_Diabetes',
       'Autoimmune_Skin/Tissues', 'Autoimmune_Others', 'Common_deficit_Ig',
       'Selective_deficit_IgA', 'Down_Syndrome', 'Dermatisis_Herpetiformis',
       'Familiarity', 'HLA_binary', 'Corazza_Histology_NUM',
       'IgA_categorical_NUM', 'EMA_categorical_NUM', 'CLASSIFICATION_NUM'],
      dtype='object')

In [63]:
Dataset.dtypes

Patient_ID                        int64
Sex                               int64
Age_Years                       float64
Growth Delay Updated              int64
Recent_Gastroenteritis            int64
BloodCount_Microcytic_Anemia      int64
GI_Upper                          int64
GI_Diarrhea                       int64
GI_WeightLoss                     int64
GI_Meteorism                      int64
GI_AbdominalPain                  int64
GI_Anorexia                       int64
GI_Stipsis                        int64
GI_Other                          int64
Neuro_Psychiatry                  int64
Neuro_Other                       int64
Neuro_Headache                    int64
Neuro_Irritability                int64
Teeth_Anomalies                   int64
Blood_Proteins_Anomalies          int64
Asthenia                          int64
Osteopenia                        int64
Autoimmune_Tyroid_Dis             int64
Autoimmune_Type1_Diabetes         int64
Autoimmune_Skin/Tissues           int64


Save the Dataset

In [64]:
Dataset.to_excel("Dataset_TO_TDA.xlsx", index=False)